# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Underag3/flyrank.ai/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**Selected Lane: Lane 2 — Refresh / Content Opportunity Scoring**

I have chosen **Lane 2: Refresh / Content Opportunity Scoring** as my provisional direction for this project. In modern organic search operations, content teams cannot afford to rewrite or monitor every page manually across a portfolio of tens of thousands of URLs. When existing pages begin losing search visibility, organic traffic declines quietly unless prioritized proactively. This lane directly addresses this operational bottleneck by building a systematic, evidence-backed scoring mechanism that ranks URLs by urgency and opportunity, surfacing which pages should be refreshed, expanded, protected, or pruned first to recover search discoverability efficiently.

In [1]:
import os, sys
import pandas as pd
import numpy as np

# Bulletproof path detection for Google Colab and Local environments
candidate_paths = [
    "../../data/raw/content_refresh_anonymized.csv",
    "data/raw/content_refresh_anonymized.csv",
    "../data/raw/content_refresh_anonymized.csv",
    "/content/flyrank.ai/data/raw/content_refresh_anonymized.csv",
    "/content/flyrank-ml-internship-starter/data/raw/content_refresh_anonymized.csv"
]

data_path = next((p for p in candidate_paths if os.path.exists(p)), None)

if data_path is not None:
    print(f"Loading dataset from local path: {data_path}")
    df = pd.read_csv(data_path)
else:
    # On Google Colab, when opened directly without manual cloning, load seamlessly via GitHub raw URL
    print("Local file not found in Colab VM. Loading directly from repository raw URL...")
    RAW_URL = "https://raw.githubusercontent.com/Underag3/flyrank.ai/main/data/raw/content_refresh_anonymized.csv"
    FALLBACK_URL = "https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv"
    try:
        df = pd.read_csv(RAW_URL)
    except Exception:
        df = pd.read_csv(FALLBACK_URL)

print(f"Loaded starter dataset: {df.shape[0]:,} rows and {df.shape[1]} columns.")
print(f"Unique clients represented: {df['client_id'].nunique()}")
print(f"Unique content items: {df['content_id'].nunique()}")

Loading dataset from local path: ../../data/raw/content_refresh_anonymized.csv
Loaded starter dataset: 30,000 rows and 44 columns.
Unique clients represented: 32
Unique content items: 30000


## 2. The question: decision, action, cost of a wrong call

- **Search Question**: Which high-value pages are suffering from organic decay or under-realized visibility, and which specific actions (refresh, expand, protect, prune) should content teams execute first to arrest traffic loss?
- **Unit of Analysis**: A single pseudonymized content item (`content_id`) within a client domain (`client_id`) evaluated over a rolling 90-day performance window.
- **Output**: A prioritized, ranked review queue where each page receives a composite opportunity score, a recommended operational action, and human-interpretable reason codes (e.g., `declining_with_demand`, `page_one_decay_risk`).
- **Decision Improved**: Weekly editorial resource allocation — deciding exactly which 10 to 20 articles a content marketing team or editorial staff should assign for content optimization rather than guessing or reacting ad hoc.
- **Action Taken**: An SEO strategist or content editor opens the flagged page, audits its topical coverage against current query intent, refreshes stale facts, enriches headings, and republishes with updated metadata.
- **Cost of a Wrong Call**:
  - *False Positive (flagging a healthy page or one with no potential)*: Wastes valuable editorial hours and writer bandwidth rewriting content that did not need updating and yields no incremental return.
  - *False Negative (missing a decaying page)*: Silently bleeds organic rank, impressions, and revenue to competing search results that capitalize on the content decay.
- **Why Data / ML is Needed**: Simple manual filters fail because real organic performance involves multi-dimensional trade-offs across impressions, position tiers, historical trend trajectory (`trend_pct`), bounce/engagement signals, and content age. A calibrated scoring model digests these noisy, interrelated signals at scale across 30,000+ pages and standardizes prioritization across different client profiles.

In [2]:
# Verify unit of analysis grain
grain_check = df.groupby(['client_id', 'content_id']).size()
print(f"Max occurrences per (client_id, content_id) pair: {grain_check.max()}")
print(f"Total distinct content items: {len(grain_check):,}")

Max occurrences per (client_id, content_id) pair: 1
Total distinct content items: 30,000


## 3. Quick look at the data (2-3 real numbers)

Below are three empirical observations from the starter dataset (`content_refresh_anonymized.csv`) that demonstrate why Lane 2 represents an urgent, high-leverage challenge worth the next 7 weeks:

1. **Majority Portfolio Decay (54.21%)**: Out of 30,000 evaluated pages, **16,262 pages (54.21%)** exhibit a downward performance trajectory (`trend_direction == 'down'`). Organic decay is not a rare edge case; it affects more than half of the active content inventory.
2. **Substantial Declining Demand (13,152 pages)**: Among pages with demonstrated search interest (`impressions_90d >= 100`), **13,152 pages (43.84%)** are actively losing traffic. For this decaying segment alone, aggregate impressions plummeted from **29,408,488** (previous 30 days) to **15,311,594** (last 30 days) — a steep drop of over **14 million impressions (-47.93%)** that could potentially be safeguarded.
3. **High-Stakes Page-One Vulnerability (7,076 pages)**: **7,076 pages (23.59%)** currently rank on the first page of Google (`avg_position <= 10`) while having aged past 180 days (`content_age_days >= 180`). These prime-position assets are prime targets for competitors if not audited for decay.

In [3]:
# 1. Downward trend share
down_count = (df['trend_direction'] == 'down').sum()
down_pct = (down_count / len(df)) * 100

# 2. Declining pages with active search demand
demand_down = df[(df['trend_direction'] == 'down') & (df['impressions_90d'] >= 100)]
demand_down_count = len(demand_down)
demand_down_pct = (demand_down_count / len(df)) * 100
imp_prev = demand_down['impressions_prev_30d'].sum()
imp_last = demand_down['impressions_last_30d'].sum()
imp_loss_pct = ((imp_last - imp_prev) / imp_prev) * 100

# 3. Page-one assets with aging decay risk
page_one_decay = df[(df['avg_position'] > 0) & (df['avg_position'] <= 10) & (df['content_age_days'] >= 180)]
page_one_count = len(page_one_decay)
page_one_pct = (page_one_count / len(df)) * 100

print(f"Number 1 - Downward trend pages: {down_count:,} / {len(df):,} ({down_pct:.2f}%)")
print(f"Number 2 - Declining with demand (>=100 imp): {demand_down_count:,} ({demand_down_pct:.2f}%)")
print(f"           30d Impression loss in this segment: {imp_prev:,} -> {imp_last:,} ({imp_loss_pct:.2f}%)")
print(f"Number 3 - Page 1 rank (pos <= 10) & aged >= 180d: {page_one_count:,} ({page_one_pct:.2f}%)")

Number 1 - Downward trend pages: 16,262 / 30,000 (54.21%)
Number 2 - Declining with demand (>=100 imp): 13,152 (43.84%)
           30d Impression loss in this segment: 29,364,746 -> 15,299,593 (-47.90%)
Number 3 - Page 1 rank (pos <= 10) & aged >= 180d: 7,076 (23.59%)


## 4. Careful words: what I can and can't claim

Adhering to rigorous empirical standards (`skills/writing-honest-claims`), this project explicitly delineates valid inferences from unjustified claims:

### What this work CAN claim:
- **Observed Patterns**: We report empirical associations present in this pseudonymized portfolio (e.g., *'pages aged over 180 days with low update frequency showed higher rates of observed impression decline'*).
- **Directional Evidence**: We describe observable trends across trailing measurement windows without asserting universal search engine dynamics.
- **Decision-Support Ranking**: We validate that the model prioritizes high-risk, high-opportunity pages more effectively than naive random or single-threshold selection at top-$K$ cutoffs (e.g., precision@20).

### What this work CANNOT claim:
- **No Causal Proof**: Observational data without randomized A/B experimentation cannot prove that updating content was the sole cause of a rank recovery.
- **No 'Predicting Google's Algorithm'**: We do not reverse-engineer Google's proprietary search ranking algorithms; we model observed portfolio metrics to assist human operational workflows.
- **No Guaranteed Traffic Outcomes**: A high refresh opportunity score indicates historical vulnerability and high exposure, not a contractual guarantee that an editor's revision will achieve a top ranking.

In [4]:
# Verification of dataset health and gotchas (per skills/flyrank/flyrank-data)
print("Data Integrity Checks:")
print(f"- Missing avg_position (value == 0): {(df['avg_position'] == 0).sum():,} rows (handled as unranked)")
print(f"- Median CTR: {df['ctr'].median():.2f}%")
print(f"- Median Days Since Last Update: {df['days_since_last_update'].median():.0f} days")
print(f"- Content Types present: {df['content_type'].nunique()} ({list(df['content_type'].dropna().unique())[:4]}...)")

Data Integrity Checks:
- Missing avg_position (value == 0): 1,205 rows (handled as unranked)
- Median CTR: 0.07%
- Median Days Since Last Update: 20 days
- Content Types present: 3 (['keyword article', 'feedly article', 'comparison article']...)


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.